# Airbnb Listings Price Prediction - Fine-Tuning Example

This notebook demonstrates how to fine-tune a deep learning model for predicting Airbnb listing prices.

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append('..')

from src.data import AirbnbDataLoader
from src.models import create_model
from src.utils import Trainer, evaluate_model, calculate_metrics, plot_predictions, plot_training_history
from src.config import Config, ModelConfig, TrainingConfig, DataConfig

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Explore Data

In [ ]:
# Create data loader
data_loader = AirbnbDataLoader()

# Generate sample data
df = data_loader.create_sample_data(n_samples=2000)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Data statistics
df.describe()

In [ ]:
# Visualize price distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(df['price'], bins=50, edgecolor='black')
plt.xlabel('Price ($)')
plt.ylabel('Frequency')
plt.title('Price Distribution')

plt.subplot(1, 2, 2)
plt.boxplot(df['price'])
plt.ylabel('Price ($)')
plt.title('Price Boxplot')

plt.tight_layout()
plt.show()

## 2. Preprocess Data

In [ ]:
# Preprocess data
preprocessed_data = data_loader.preprocess_data(
    df,
    target_column='price',
    test_size=0.2,
    val_size=0.1
)

print(f"Train samples: {len(preprocessed_data['train'][0])}")
print(f"Val samples: {len(preprocessed_data['val'][0])}")
print(f"Test samples: {len(preprocessed_data['test'][0])}")
print(f"\nInput features: {preprocessed_data['train'][0].shape[1]}")

In [ ]:
# Create dataloaders
dataloaders = data_loader.create_dataloaders(
    preprocessed_data,
    batch_size=32,
    shuffle=True
)

print("DataLoaders created successfully!")

## 3. Create and Train Model

In [ ]:
# Model configuration
input_dim = preprocessed_data['train'][0].shape[1]
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {device}")

# Create model
model = create_model(
    input_dim=input_dim,
    model_type='simple',
    hidden_dims=[128, 64, 32],
    dropout=0.2
)

print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Setup training
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# Create trainer
trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    scheduler=scheduler
)

print("Trainer created successfully!")

In [ ]:
# Train the model
history = trainer.fit(
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    epochs=30,
    save_path='best_model.pth',
    early_stopping_patience=10
)

## 4. Visualize Training Progress

In [ ]:
# Plot training history
plot_training_history(history)

## 5. Evaluate Model

In [ ]:
# Load best model
trainer.load_checkpoint('best_model.pth')

# Evaluate on test set
predictions, targets = evaluate_model(model, dataloaders['test'], device)

# Calculate metrics
metrics = calculate_metrics(targets, predictions)

print("Test Set Metrics:")
print("=" * 40)
for metric, value in metrics.items():
    print(f"{metric:10s}: {value:.4f}")

In [ ]:
# Plot predictions vs actual
plot_predictions(targets, predictions, title='Test Set: Predictions vs Actual')

## 6. Fine-Tuning Example (Transfer Learning)

In [ ]:
# Save the base model for transfer learning
torch.save(model.state_dict(), 'base_model.pth')
print("Base model saved!")

In [ ]:
from src.utils import FineTuner

# Create a new model with pretrained base
pretrained_model = create_model(
    input_dim=input_dim,
    model_type='pretrained',
    freeze_base=True
)

print(f"Total parameters: {sum(p.numel() for p in pretrained_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad):,}")

In [ ]:
# Setup fine-tuning
criterion_ft = nn.MSELoss()
optimizer_ft = optim.Adam(pretrained_model.parameters(), lr=0.001)
scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(optimizer_ft, mode='min', patience=3)

# Create fine-tuner
finetuner = FineTuner(
    model=pretrained_model,
    criterion=criterion_ft,
    optimizer=optimizer_ft,
    device=device,
    scheduler=scheduler_ft
)

In [ ]:
# Fine-tune with progressive unfreezing
ft_history = finetuner.progressive_unfreezing(
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    phases=[(5, True), (10, False)],  # 5 epochs frozen, 10 epochs unfrozen
    save_path='finetuned_model.pth'
)

In [ ]:
# Plot fine-tuning history
plot_training_history(ft_history)

In [ ]:
# Evaluate fine-tuned model
ft_predictions, ft_targets = evaluate_model(pretrained_model, dataloaders['test'], device)
ft_metrics = calculate_metrics(ft_targets, ft_predictions)

print("Fine-Tuned Model Metrics:")
print("=" * 40)
for metric, value in ft_metrics.items():
    print(f"{metric:10s}: {value:.4f}")

## 7. Compare Models

In [ ]:
import pandas as pd

# Create comparison dataframe
comparison = pd.DataFrame({
    'Metric': list(metrics.keys()),
    'Base Model': list(metrics.values()),
    'Fine-Tuned Model': list(ft_metrics.values())
})

print("\nModel Comparison:")
print(comparison.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Base model
axes[0].scatter(targets, predictions, alpha=0.5, label='Predictions')
axes[0].plot([targets.min(), targets.max()], [targets.min(), targets.max()], 'r--', lw=2, label='Perfect')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title(f'Base Model (R² = {metrics["R2"]:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Fine-tuned model
axes[1].scatter(ft_targets, ft_predictions, alpha=0.5, label='Predictions', color='green')
axes[1].plot([ft_targets.min(), ft_targets.max()], [ft_targets.min(), ft_targets.max()], 'r--', lw=2, label='Perfect')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].set_title(f'Fine-Tuned Model (R² = {ft_metrics["R2"]:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated:
1. Loading and preprocessing Airbnb listings data
2. Training a neural network for price prediction
3. Fine-tuning with transfer learning and progressive unfreezing
4. Evaluating and comparing models

The fine-tuning approach allows you to:
- Leverage pretrained representations
- Reduce training time
- Potentially achieve better performance with less data